In [ ]:
#@title 🌸 Animagine XL 4.0 Opt — Kaggle T4 x2
# Kaggle: เปิด Internet และเลือก Accelerator เป็น GPU T4 x2 ก่อนรัน
# ติดตั้งส่วนเสริมโดยไม่อัปเกรด torch ที่ Kaggle เตรียม CUDA มาให้
!pip install -q diffusers transformers accelerate gradio safetensors

import random
import gradio as gr
import torch
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU — ไปที่ Kaggle Notebook Settings แล้วเลือก Accelerator: GPU T4 x2')

gpu_count = torch.cuda.device_count()
print(f'พบ GPU {gpu_count} ตัว: ' + ', '.join(torch.cuda.get_device_name(i) for i in range(gpu_count)))
print('หมายเหตุ: VRAM ของ T4 สองตัวไม่ถูกรวมเป็นก้อนเดียว; งานสร้างภาพนี้ใช้ GPU 0')

model_id = 'cagliostrolab/animagine-xl-4.0-opt'
pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
pipe = pipe.to('cuda:0')

def generate_anime_v4(prompt, negative_prompt, aspect_ratio, steps, cfg_scale, seed):
    sizes = {
        'จัตุรัส (1:1)': (1024, 1024),
        'แนวตั้งมาตรฐาน (3:4)': (896, 1152),
        'แนวตั้ง (9:16 - มือถือ)': (832, 1216),
        'แนวนอน (16:9 - หน้าจอคอม)': (1216, 832),
    }
    width, height = sizes.get(aspect_ratio, (896, 1152))
    actual_seed = random.randint(0, 2147483647) if seed is None or int(seed) == -1 else int(seed)
    generator = torch.Generator(device='cuda:0').manual_seed(actual_seed)
    try:
        with torch.inference_mode():
            image = pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                width=width,
                height=height,
                num_inference_steps=int(steps),
                guidance_scale=float(cfg_scale),
                generator=generator,
            ).images[0]
        return image, actual_seed
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise gr.Error('VRAM ไม่พอ: ลดขนาดภาพหรือ Steps แล้วลองอีกครั้ง') from exc

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🌸 AI วาดภาพอนิเมะ — Animagine XL 4.0 Opt (Kaggle T4 x2)')
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label='Prompt (คำสั่งวาดภาพ)', lines=3,
                value='1girl, silver hair, blue eyes, stylish casual clothes, sunset city background, masterpiece, high score, great score, absurdres',
            )
            neg_prompt_input = gr.Textbox(
                label='Negative Prompt (สิ่งที่ไม่ต้องการ)', lines=3,
                value='lowres, bad anatomy, bad hands, text, error, missing fingers, extra digits, fewer digits, cropped, worst quality, low quality, low score, bad score, average score',
            )
            aspect_dropdown = gr.Dropdown(
                choices=['จัตุรัส (1:1)', 'แนวตั้งมาตรฐาน (3:4)', 'แนวตั้ง (9:16 - มือถือ)', 'แนวนอน (16:9 - หน้าจอคอม)'],
                value='แนวตั้งมาตรฐาน (3:4)', label='สัดส่วนภาพ (Aspect Ratio)',
            )
            with gr.Row():
                steps_slider = gr.Slider(15, 40, value=28, step=1, label='Steps (แนะนำ 25–30)')
                cfg_slider = gr.Slider(1.0, 10.0, value=5.0, step=0.5, label='CFG Scale (แนะนำ 4.0–5.0)')
            seed_input = gr.Number(value=-1, label='Seed (-1 = สุ่มใหม่ทุกครั้ง)', precision=0)
            generate_btn = gr.Button('✨ วาดภาพอนิเมะ', variant='primary')
        with gr.Column():
            output_image = gr.Image(label='ผลลัพธ์')
            used_seed_output = gr.Number(label='Seed ที่ใช้สร้างภาพนี้', interactive=False)
    generate_btn.click(
        fn=generate_anime_v4,
        inputs=[prompt_input, neg_prompt_input, aspect_dropdown, steps_slider, cfg_slider, seed_input],
        outputs=[output_image, used_seed_output],
    )

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)